 LangChain1.x 的短期记忆是三者的组合：
 
**State（会话内部状态） + Checkpointer（持久化机制） + Thread ID（会话作用域）**

- **State**：默认存储历史消息列表`messages`，通过State 管理历史消息
- **Checkpointer**：负责将State 作为检查点持久化保存，检查点是某个时刻的State 快照
- **Thread ID**：用于唯一标识State，LangChain运行时会按照 `thread_id` 读写State快照

> 这就像玩 RPG 游戏时的“自动存档”：你不需要手动保存，系统在关键节点自动记录，下次进入游戏随时可以从上次的存档点继续。

## 举例1：没有记忆

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model = "gpt-5.4-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[]
)


print("\n第一轮对话: ")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话: ")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话: 
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的？

第二轮对话: 
Agent: 我不知道你的名字，除非你告诉我。  
如果你愿意，可以直接发我你的名字，我就这样称呼你。


## 举例2：有记忆

注意我们没有显式的维护消息列表，而是LangChain通过 checkpointer 来管理。
checkpointer 会自动保存和恢复消息，确保在不同调用之间保持上下文。这是框架与手搓的区别。

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 1. 创建 Agent 时添加 checkpointer
agent = create_agent(
    model=model,
    checkpointer=checkpointer  # 添加内存管理
)

# 2. 调用时指定 thread_id
config = {
    "configurable": {
        "thread_id": "1"
    }
}

print("\n第一轮对话: ")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config  # 传入 config
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话: ")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话: 
Agent: 你好，张三！很高兴认识你。  
我是你的 AI 助手，有什么我可以帮你的吗？

第二轮对话: 
Agent: 你叫张三。


In [ ]:
from rich import print as rprint
latest_state = agent.get_state(config)
rprint(latest_state) # 查看完整状态图快照，打印StateSnapshot实例，包含当前状态值values，下一个节点属性next等

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='af1288cb-b2a6-4a79-bfcc-f5b0c8cf9722'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。  \n我是你的 AI 助手，有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 30,
                        'prompt_tokens': 10,
                        'total_tokens': 40,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
                        'cost': 0.0001425,
                        'is_byok': False,
                        'cost_details': {
                            'upstream_inference_cost': 0.0001425,
                            'upstream_inference_prompt_cost': 7.5e-06,
                            'upstream_inference_completions_cost': 0.000135
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'openai/gpt-5.4-mini',
                    'system_fingerprint': None,
                    'id': 'gen-1785397635-IdWI1sdoqRMdb4XVzhw3',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019fb1fd-c6e5-72f3-85e1-7262d3c5a674-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 10,
                    'output_tokens': 30,
                    'total_tokens': 40,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 0}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='a42b98d4-0805-4375-b49d-8a3f4bccce2c'
            ),
            AIMessage(
                content='你叫张三。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 9,
                        'prompt_tokens': 49,
                        'total_tokens': 58,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
                        'cost': 7.725e-05,
                        'is_byok': False,
                        'cost_details': {
                            'upstream_inference_cost': 7.725e-05,
                            'upstream_inference_prompt_cost': 3.675e-05,
                            'upstream_inference_completions_cost': 4.05e-05
                        }
              

In [5]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：“我叫什么？”


In [ ]:
from rich import print as rprint
latest_state = agent.get_state(config)
rprint(latest_state) # 之前的对话会更新状态快照

In [6]:
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}
response4 = agent.invoke(
    {"messages": [HumanMessage("你还记得我叫什么名字么？")]},
    config=config2
)
print(response4['messages'][-1].content)

我现在**不记得**，因为我在这段对话里没有看到你的名字记录。

如果你愿意，可以告诉我，我这次就记住你怎么称呼你。


## 关键解析

第1步：初始化记忆引擎：`checkpointer = InMemorySaver()`——创建一个内存级的记忆存储。
> 注意：InMemorySaver内存中保存，进程结束就丢失数据，适合测试。生产环境可换成数据库持久化的 `SqliteSaver`、`PostgresSaver` 等

第2步：绑定 Agent：在 `create_agent` 时传入 `checkpointer`，让 Agent 具备状态存储能力。

第3步：设定会话 ID：通过 `config = {"configurable": {"thread_id": "1"}}` 为每次调用指定线程标识。
同一个 thread_id 共享记忆，不同 thread_id 完全隔离。
```python
# 会话 1
config1 = {"configurable": {"thread_id": "1"}}
agent.invoke({...}, config=config1)

# 会话 2
config2 = {"configurable": {"thread_id": "2"}}
agent.invoke({...}, config=config2)

# 两个会话完全独立
```
`thread_id` 是记忆管理的核心开关：在会话2里询问会话1的会话信息，Agent 会表示不知道——因为双方记忆空间完全隔离。

## 场景示例

### 场景1：多用户聊天
不同 thread_id = 不同会话，Agent 能正确记住每个会话的内容。
```python
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver()
)


# 用户 Alice
config_alice = {"configurable": {"thread_id": "user_alice"}}
agent.invoke({"messages": [...]}, config_alice)
...

# 用户 Bob
config_bob = {"configurable": {"thread_id": "user_bob"}}
agent.invoke({"messages": [...]}, config_bob)
...

# 两个会话完全独立
```

### 场景2：同一用户的不同任务
```python
# 任务 1：写代码
config_task1 = {"configurable": {"thread_id": "task_coding"}}
agent.invoke({"messages": [...]}, config_task1)
...

# 任务 2：写文档
config_task2 = {"configurable": {"thread_id": "task_docs"}}
agent.invoke({"messages": [...]}, config_task2)
...
```

## 工作原理
### 内存保存了什么？
```python
agent.invoke({"messages": [{"role": "user", "content": "你好"}]}, config)
# InMemorySaver 保存：
# {
#   "thread_id": "xxx",
#   "messages": [
#       HumanMessage("你好"),
#       AIMessage("你好！有什么可以帮助你的吗？")
#   ]
# }

agent.invoke({"messages": [{"role": "user", "content": "天气"}]}, config)
# InMemorySaver 更新：
# {
#   "thread_id": "xxx",
#   "messages": [
#       HumanMessage("你好"),
#       AIMessage("你好！有什么可以帮助你的吗？"),
#       HumanMessage("天气"),
#       AIMessage("...")
#   ]
# }
```
### 自动追加历史
```python
# 你只需要传新消息
agent.invoke(
    {"messages": [{"role": "user", "content": "新问题"}]},
    config
)
```
此时，checkpointer 自动：
① 读取之前的历史
② 追加新消息
③ 调用模型（传入完整历史）
④ 保存新的历史

说明：checkpointer 会自动管理历史

## 常见问题
### 1、为什么 Agent 不记得？
检查：
- 是否添加了 `checkpointer=InMemorySaver()`？
- 是否传入了 `config` 参数？
- 两次调用的 `thread_id` 是否相同？

```python
# ❌ 错误：没有 checkpointer
agent = create_agent(model=model, tools=[])
agent.invoke({...})  # 不会记住

# ❌ 错误：没有 config
agent = create_agent(model=model, tools=[], checkpointer=InMemorySaver())
agent.invoke({...})  # 不会记住

# ❌ 错误：thread_id 不同
agent.invoke({...}, config={"configurable": {"thread_id": "1"}})
agent.invoke({...}, config={"configurable": {"thread_id": "2"}}) # 不同会话

# ✅ 正确
agent = create_agent(model=model, tools=[], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}
agent.invoke({...}, config)
agent.invoke({...}, config)  # 记得！
```

### 2、InMemorySaver 会丢失数据吗？
会！InMemorySaver 只保存在内存中：
- ✅ 进程内有效（不支持跨进程共享）
- ❌ 程序重启后丢失（或进程重启后丢失）
- ❌ 不同进程无法共享
解决方案：持久化（SQLite、PostgreSQL）

### 3、内存会无限增长吗？
会！默认情况下，InMemorySaver 会保存所有消息。
问题：
- 消息越来越多（无限增长，需要管理上下文）
- token越来越多，甚至会超过模型的 token 限制
- 响应速度变慢、成本增加
解决方案：上下文管理（修剪、摘要）

### 4、如何清空某个会话的历史？
目前 InMemorySaver 没有提供删除API。
临时方案：
- 使用新的 `thread_id`
- 或重新创建 Agent
